In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
spark = SparkSession.builder.appName("Skripsi").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/28 19:38:56 WARN Utils: Your hostname, pc, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/28 19:38:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/28 19:38:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
df = spark.read.csv("dataset.csv", header=True, inferSchema=True)

valid_temps = [-20, -10, 0, 10, 25, 40]

closest_val = F.lit(valid_temps[0])
min_diff = F.abs(F.col("Ambient_Temp_degC") - F.lit(valid_temps[0]))

for t in valid_temps[1:]:
    current_diff = F.abs(F.col("Ambient_Temp_degC") - F.lit(t))
    
    closest_val = F.when(current_diff < min_diff, F.lit(t)).otherwise(closest_val)
    min_diff = F.when(current_diff < min_diff, current_diff).otherwise(min_diff)

df = df.withColumn("Ambient_Temp_degC", closest_val)

In [6]:
total_rows = df.count()
total_unique_rows = df.dropDuplicates().count()

duplicate_count = total_rows - total_unique_rows
print(f"Total duplicate rows: {duplicate_count}")

Total duplicate rows: 0


In [10]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.partitionBy(
    "Test_Cell",
    "Ambient_Temp_degC",
    "TimeStamp"
)

df_check = (
    df.withColumn("duplicate_count", F.count("*").over(w))
      .withColumn(
          "is_duplicate",
          F.col("duplicate_count") > 1
      )
)

df_check.show(truncate=False)

+-----------------------+-------+----------------+---------+-----+----------------+-------+-------+-----------------+-----------------+------------+---------+---------------+------------+
|TimeStamp              |Time_s |Voltage_V       |Current_A|Ah   |SOC             |Power_W|Wh     |Ambient_Temp_degC|Battery_Temp_degC|Cycle_Label |Test_Cell|duplicate_count|is_duplicate|
+-----------------------+-------+----------------+---------+-----+----------------+-------+-------+-----------------+-----------------+------------+---------+---------------+------------+
|10/11/2021 06:52:52.000|4603020|3.396           |0.0      |3.627|0.15582365402566|0.0    |12.2833|-20              |1.0435           |CC_CV_charge|m1000    |2              |true        |
|10/11/2021 06:52:52.000|5135623|3.396           |0.0      |3.627|0.15582365402566|0.0    |12.2833|-20              |1.0435           |CC_CV_charge|m1000    |2              |true        |
|10/11/2021 06:53:18.000|4603046|3.39589949849549|0.0      |

In [7]:
df.show()

+--------------------+------+----------------+-----------------+------+-----------------+-----------------+------+-----------------+-----------------+------------+---------+
|           TimeStamp|Time_s|       Voltage_V|        Current_A|    Ah|              SOC|          Power_W|    Wh|Ambient_Temp_degC|Battery_Temp_degC| Cycle_Label|Test_Cell|
+--------------------+------+----------------+-----------------+------+-----------------+-----------------+------+-----------------+-----------------+------------+---------+
|10/29/2021 16:25:...|     0|2.78867142857143|          -2.0959|3.7508|0.123643079252387|-5.84305819571432|12.825|             25.0|  5.6138285714286|CC_CV_charge|    m1000|
|10/29/2021 16:25:...|     1|2.80099957862128|-2.24341190186706|3.7508|0.123643079252387|-6.28110464284739|12.825|             25.0| 5.59957608967206|CC_CV_charge|    m1000|
|10/29/2021 16:25:...|     2|2.80642753618361|-2.20590919538316|3.7508|0.123643079252387|-6.17610456523376|12.825|             25.